<a href="https://colab.research.google.com/github/lianafarsi/medical-ner-extractor/blob/main/medical_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle -q

from google.colab import files
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d tboyle18/medical-insurance-cost

!kaggle datasets download -d darshan1504/medical-transcription

Saving kaggle.json to kaggle.json
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


In [2]:
import pandas as pd
import re
from transformers import pipeline

In [3]:
def clean_medical_text(text: str) -> str:
    """
    Cleans raw medical text by removing HTML tags and extra whitespaces.
    """
    if not isinstance(text, str):
        return ""

    # Remove HTML tags if present in the text
    text = re.sub(r'<.*?>', '', text)

    # Normalize multiple whitespace characters into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [4]:
def run_medical_ner(texts: list):
    """
    Initializes the Biomedical NER pipeline and extracts medical entities.
    """
    print("Initializing Biomedical NER pipeline...")
    ner_pipeline = pipeline("ner", model="d4data/biomedical-ner-all", tokenizer="d4data/biomedical-ner-all")

    all_results = []
    for idx, text in enumerate(texts):
        print(f"\n--- Processing Text #{idx + 1} ---")
        print(f"Text snippet: {text[:150]}...")

        entities = ner_pipeline(text)
        all_results.append({"text_id": idx, "entities": entities})

        for entity in entities:
            print(f"  -> Entity: {entity['word']} | Type: {entity['entity']} | Confidence: {entity['score']:.4f}")

    return all_results

In [5]:
# Sample mock dataset for testing before loading the actual Kaggle CSV file
sample_data = [
    "Patient presents with severe headache and acute migraine. Prescribed Sumatriptan 50mg daily.",
    "History of type 2 diabetes. The patient was advised to take Metformin 500mg twice a day."
]

# Step 1: Preprocess the texts
cleaned_texts = [clean_medical_text(t) for t in sample_data]

# Step 2: Run the NER extraction model
extraction_results = run_medical_ner(cleaned_texts)

Initializing Biomedical NER pipeline...


config.json:   0%|          | 0.00/5.00k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  266MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]


--- Processing Text #1 ---
Text snippet: Patient presents with severe headache and acute migraine. Prescribed Sumatriptan 50mg daily....
  -> Entity: severe | Type: B-Severity | Confidence: 0.9998
  -> Entity: headache | Type: B-Sign_symptom | Confidence: 0.9999
  -> Entity: acute | Type: B-Detailed_description | Confidence: 0.9999
  -> Entity: mig | Type: B-Sign_symptom | Confidence: 0.9999
  -> Entity: sum | Type: B-Medication | Confidence: 0.9998
  -> Entity: ##at | Type: B-Medication | Confidence: 0.7563
  -> Entity: ##rip | Type: I-Medication | Confidence: 0.9893
  -> Entity: ##tan | Type: I-Medication | Confidence: 0.9961
  -> Entity: 50 | Type: B-Dosage | Confidence: 0.9995
  -> Entity: ##mg | Type: I-Dosage | Confidence: 0.9997

--- Processing Text #2 ---
Text snippet: History of type 2 diabetes. The patient was advised to take Metformin 500mg twice a day....
  -> Entity: type | Type: B-History | Confidence: 0.9804
  -> Entity: 2 | Type: I-History | Confidence: 0.9867
  -> Ent

In [6]:
import pandas as pd

def parse_ner_results(extraction_results: list) -> pd.DataFrame:
    """
    Parses the raw NER model output and converts it into a structured pandas DataFrame
    for downstream analysis and storage.
    """
    structured_rows = []

    for item in extraction_results:
        text_id = item['text_id']
        for entity in item['entities']:
            # Extracting relevant keys safely
            structured_rows.append({
                "text_id": text_id,
                "entity_text": entity.get('word'),
                "entity_type": entity.get('entity'),
                "confidence_score": round(entity.get('score', 0.0), 4),
                "start_index": entity.get('start'),
                "end_index": entity.get('end')
            })

    # Convert list of dictionaries into a clean Pandas DataFrame
    df_structured = pd.DataFrame(structured_rows)
    return df_structured

# Execution block (Using the results from the previous step)
structured_df = parse_ner_results(extraction_results)

# Display the structured table
print("\n--- Structured Output DataFrame ---")
display(structured_df)


--- Structured Output DataFrame ---


,text_id,entity_text,entity_type,confidence_score,start_index,end_index
0,0,severe,B-Severity,0.9998,22,28
1,0,headache,B-Sign_symptom,0.9999,29,37
2,0,acute,B-Detailed_description,0.9999,42,47
3,0,mig,B-Sign_symptom,0.9999,48,51
4,0,sum,B-Medication,0.9998,69,72
5,0,##at,B-Medication,0.7563,72,74
6,0,##rip,I-Medication,0.9893,74,77
7,0,##tan,I-Medication,0.9961,77,80
8,0,50,B-Dosage,0.9995,81,83
9,0,##mg,I-Dosage,0.9997,83,85


In [7]:
def filter_and_validate_entities(df: pd.DataFrame, min_confidence: float = 0.85) -> pd.DataFrame:
    """
    Filters out low-confidence predictions and removes duplicate entity entries
    to ensure high data quality for clinical downstream tasks.
    """
    if df.empty:
        print("DataFrame is empty. No entities to filter.")
        return df

    # Step 1: Filter by minimum confidence threshold
    filtered_df = df[df['confidence_score'] >= min_confidence].copy()

    # Step 2: Remove exact duplicate rows if the same entity was captured multiple times
    filtered_df = filtered_df.drop_duplicates(subset=['text_id', 'entity_text', 'entity_type']).reset_index(drop=True)

    print(f"Original entities count: {len(df)}")
    print(f"Filtered entities count (Confidence >= {min_confidence}): {len(filtered_df)}")

    return filtered_df

# Execution block (Using the structured DataFrame from the previous step)
validated_df = filter_and_validate_entities(structured_df, min_confidence=0.80)

# Display the final clean table
display(validated_df)

Original entities count: 19
Filtered entities count (Confidence >= 0.8): 17


,text_id,entity_text,entity_type,confidence_score,start_index,end_index
0,0,severe,B-Severity,0.9998,22,28
1,0,headache,B-Sign_symptom,0.9999,29,37
2,0,acute,B-Detailed_description,0.9999,42,47
3,0,mig,B-Sign_symptom,0.9999,48,51
4,0,sum,B-Medication,0.9998,69,72
5,0,##rip,I-Medication,0.9893,74,77
6,0,##tan,I-Medication,0.9961,77,80
7,0,50,B-Dosage,0.9995,81,83
8,0,##mg,I-Dosage,0.9997,83,85
9,1,type,B-History,0.9804,11,15


In [8]:
import os

def export_processed_data(df: pd.DataFrame, output_filename: str = "extracted_medical_entities.csv"):
    """
    Exports the validated medical entities DataFrame to a CSV file
    and ensures the output directory exists for professional project structure.
    """
    # Create an 'output' directory if it doesn't exist (Engineering Best Practice)
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)

    file_path = os.path.join(output_dir, output_filename)

    # Save DataFrame to CSV without index to keep it clean
    df.to_csv(file_path, index=False)
    print(f"Successfully exported data to: {file_path}")

    return file_path

# Execution block
saved_file_path = export_processed_data(validated_df)

Successfully exported data to: output/extracted_medical_entities.csv
